In [ ]:
"""
AuraFlow LLM Fine-Tuning - Environment Setup (Colab)

This cell prepares the Google Colab environment for Large Language Model (LLM) 
fine-tuning. It resolves specific dependency conflicts (particularly with 
quantization libraries) and installs a validated set of library versions 
(transformers, peft, trl) known to work together stably.

Note: I trained the model in Chinese, so there might remain Chinese in the running results.
"""

import os

# 1. Hardware Verification
# Check GPU availability and status to ensure training can proceed
!nvidia-smi

# 2. Dependency Resolution: bitsandbytes
# The default bitsandbytes version in Colab may conflict with modern quantization methods.
# We perform a clean reinstall to ensure compatibility with the torch stack.
!echo "--- Reconfiguring bitsandbytes environment... ---"
!pip uninstall -y bitsandbytes

# Upgrade pip to handle dependency resolution better
!echo "--- Upgrading pip... ---"
!python -m pip install --upgrade pip

# Install the stable version of bitsandbytes
!echo "--- Installing stable bitsandbytes... ---"
!pip install --upgrade bitsandbytes

# 3. Install Core SFT Dependencies
# We install a specific set of pinned versions ("Golden Combination") for 
# Transformers, PEFT, and TRL. These versions are verified to prevent 
# compatibility issues during the Supervised Fine-Tuning (SFT) process.
!echo "--- Installing SFT (Supervised Fine-Tuning) libraries... ---"
!pip install \
  "transformers==4.38.2" \
  "peft==0.9.0" \
  "trl==0.7.11" \
  "accelerate==0.27.2" \
  "datasets==2.18.0" \
  safetensors \
  sentencepiece \
  tokenizers \
  wandb \
  "diffusers==0.27.2" \
  einops \
  "huggingface-hub==0.20.3"

# 4. Post-Installation Instructions
print("="*60)
print("Environment setup complete.")
print("ACTION REQUIRED: Please go to 'Runtime -> Restart session' in the menu.")
print("After restarting, SKIP this cell and proceed directly to Cell 2.")
print("="*60)

/bin/bash: line 1: nvidia-smi: command not found
--- 正在卸载 Colab 默认的 bitsandbytes (dev) ... ---
--- 正在升级 pip... ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
--- 正在安装最新稳定版 bitsandbytes... ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 50.6 MB/s  0:00:01
--- 正在安装 SFT 库... ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 26.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 44.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.1 MB/s  0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.35.3
    Uninstalling huggingface-hub-0.35.3:
      Success

In [ ]:
"""
Cell 2: Authentication

This cell handles logging into Hugging Face (for model access) and 
Weights & Biases (for experiment tracking).
"""
import huggingface_hub
import wandb

# --- Hugging Face Login ---
# Authentication is required to download gated models (e.g., Qwen/Llama)
# and push trained adapters to the hub.
print("--- Logging into Hugging Face ---")
huggingface_hub.notebook_login()

# --- Weights & Biases (W&B) Login ---
# Authenticate with W&B to enable real-time loss logging and 
# visualization of training metrics.
print("\n--- Logging into Weights & Biases (W&B) ---")
wandb.login()

--- 正在登录 Hugging Face ---



--- 正在登录 Weights & Biases (W&B) ---


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: caressez15 (caressez15-university-of-sydney-union) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
"""
Cell 3: SFT Data Preparation and Splitting

This cell defines the instruction dataset for Supervised Fine-Tuning (SFT),
converts it into a Hugging Face Dataset object, and performs a train/eval split
to monitor model performance and prevent overfitting.
"""
import json
from datasets import load_dataset, Dataset
import os

# --- SFT Dataset Definition ---
# The dataset contains instruction-response pairs for the model to learn specific behaviors.
# (Ensure the full list of data entries is included below)
sft_data = [
    # --- Category 1: 基础情绪 (Simple Emotions) ---
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要一段快乐的背景音乐。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'major', 'tempo': '120bpm', 'instruments': 'ukulele, acoustic guitar, light percussion', 'genre': 'acoustic folk', 'feel': 'bright, cheerful, upbeat'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "来点悲伤的音乐。"},
        {"role": "assistant", "content": "{'key': 'A minor', 'mode': 'minor', 'tempo': '65bpm', 'instruments': 'solo piano, slow strings', 'genre': 'cinematic', 'feel': 'somber, melancholic, reflective'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我感觉很平静，很放松。"},
        {"role": "assistant", "content": "{'key': 'F Major', 'mode': 'major', 'tempo': '60bpm', 'instruments': 'ambient synth pad, gentle electric piano', 'genre': 'ambient', 'feel': 'calm, serene, meditative'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "愤怒的，非常有力的。"},
        {"role": "assistant", "content": "{'key': 'D minor', 'mode': 'minor', 'tempo': '140bpm', 'instruments': 'distorted electric guitar, heavy drums, orchestral hits', 'genre': 'rock', 'feel': 'angry, intense, powerful'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要焦虑不安的BGM。"},
        {"role": "assistant", "content": "{'key': 'G# minor', 'mode': 'minor', 'tempo': '130bpm', 'instruments': 'fast pizzicato strings, dissonant chords', 'genre': 'cinematic suspense', 'feel': 'anxious, tense, unsettling'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "充满希望的，光明的。"},
        {"role": "assistant", "content": "{'key': 'G Major', 'mode': 'major', 'tempo': '110bpm', 'instruments': 'string orchestra, flute, harp', 'genre': 'orchestral', 'feel': 'hopeful, uplifting, bright'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "浪漫的感觉。"},
        {"role": "assistant", "content": "{'key': 'E-flat Major', 'mode': 'major', 'tempo': '80bpm', 'instruments': 'romantic piano, warm strings, saxophone', 'genre': 'jazz ballad', 'feel': 'romantic, intimate, warm'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "神秘的，悬疑的。"},
        {"role": "assistant", "content": "{'key': 'C# minor', 'mode': 'minor', 'tempo': '90bpm', 'instruments': 'synth pad, music box, reverse cymbals', 'genre': 'ambient suspense', 'feel': 'mysterious, eerie, curious'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "怀旧的，温暖的。"},
        {"role": "assistant", "content": "{'key': 'D Major', 'mode': 'major', 'tempo': '85bpm', 'instruments': 'acoustic guitar, harmonica, brushed drums', 'genre': 'folk', 'feel': 'nostalgic, warm, gentle'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "孤独的，空旷的。"},
        {"role": "assistant", "content": "{'key': 'F minor', 'mode': 'minor', 'tempo': '55bpm', 'instruments': 'solo cello, distant piano, light wind sound', 'genre': 'ambient', 'feel': 'lonely, empty, desolate'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "俏皮的，好玩的。"},
        {"role": "assistant", "content": "{'key': 'G Major', 'mode': 'major', 'tempo': '130bpm', 'instruments': 'pizzicato strings, xylophone, clarinet', 'genre': 'comedy', 'feel': 'playful, quirky, fun'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "史诗感的，宏大的。"},
        {"role": "assistant", "content": "{'key': 'D Major', 'mode': 'major', 'tempo': '140bpm', 'instruments': 'full orchestra, epic choir, taiko drums', 'genre': 'epic orchestral', 'feel': 'grand, heroic, majestic'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "困惑的，迷茫的。"},
        {"role": "assistant", "content": "{'key': 'A minor', 'mode': 'dorian', 'tempo': '100bpm', 'instruments': 'electric piano, synth arpeggio, reversed sounds', 'genre': 'electronic', 'feel': 'confused, lost, searching'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "感恩的，感动的。"},
        {"role": "assistant", "content": "{'key': 'F Major', 'mode': 'major', 'tempo': '70bpm', 'instruments': 'string quartet, piano', 'genre': 'cinematic', 'feel': 'grateful, touching, heartfelt'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "胜利的，凯旋的。"},
        {"role": "assistant", "content": "{'key': 'B-flat Major', 'mode': 'major', 'tempo': '130bpm', 'instruments': 'brass fanfare, triumphant strings, snare drum', 'genre': 'orchestral', 'feel': 'victorious, triumphant, celebratory'}"}
    ]},

    # --- 类别 2: 复合情绪 (Complex Emotions) ---
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要那种... 紧张但又有点暧昧的感觉。"},
        {"role": "assistant", "content": "{'key': 'D minor', 'mode': 'phrygian', 'tempo': '120bpm', 'instruments': 'pizzicato strings, low synth bass, subtle high-hat', 'genre': 'suspense', 'feel': 'tense, ambiguous, seductive'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "悲伤，但又带着一丝希望。"},
        {"role": "assistant", "content": "{'key': 'G minor', 'mode': 'minor', 'tempo': '75bpm', 'instruments': 'piano melody, underlying string pad, flute counter-melody', 'genre': 'cinematic', 'feel': 'melancholic but hopeful, bittersweet'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "平静中带着一点点不安。"},
        {"role": "assistant", "content": "{'key': 'E minor', 'mode': 'minor', 'tempo': '60bpm', 'instruments': 'ambient pad, single piano notes, distant dissonant chime', 'genre': 'ambient', 'feel': 'calm but unsettling, uneasy'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "怀旧的，但是快乐的童年回忆。"},
        {"role": "assistant", "content": "{'key': 'F Major', 'mode': 'major', 'tempo': '100bpm', 'instruments': 'music box, glockenspiel, light acoustic guitar', 'genre': 'acoustic', 'feel': 'nostalgic, happy, innocent'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一种下定决心的，严肃的氛围。"},
        {"role": "assistant", "content": "{'key': 'C minor', 'mode': 'minor', 'tempo': '110bpm', 'instruments': 'strong string ostinato, timpani, low brass', 'genre': 'cinematic', 'feel': 'determined, serious, focused'}"}
    ]},
     {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "喜悦的，但不是那种很闹的，是发自内心的喜悦。"},
        {"role": "assistant", "content": "{'key': 'D Major', 'mode': 'major', 'tempo': '90bpm', 'instruments': 'warm string quartet, gentle piano arpeggios', 'genre': 'chamber music', 'feel': 'joyful, heartfelt, serene'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一种很酷的，自信的感觉。"},
        {"role": "assistant", "content": "{'key': 'E minor', 'mode': 'minor', 'tempo': '100bpm', 'instruments': 'electric guitar riff, solid drum beat, synth bass', 'genre': 'rock', 'feel': 'cool, confident, edgy'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "疲惫但满足。"},
        {"role": "assistant", "content": "{'key': 'A Major', 'mode': 'major', 'tempo': '70bpm', 'instruments': 'electric piano (Rhodes), soft acoustic guitar, light brushed drums', 'genre': 'jazz', 'feel': 'tired, content, relaxed'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "神圣的，庄严的。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'major', 'tempo': '60bpm', 'instruments': 'pipe organ, church choir pad, tubular bells', 'genre': 'classical', 'feel': 'sacred, solemn, majestic'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "紧张的，像在倒计时。"},
        {"role": "assistant", "content": "{'key': 'F# minor', 'mode': 'minor', 'tempo': '135bpm', 'instruments': 'ticking clock sound, fast string staccato, low synth pulse', 'genre': 'suspense', 'feel': 'tense, urgent, countdown'}"}
    ]},

    # --- Category 3: Vlogger/UP主 场景 (Creator Scenarios) ---
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我正在做一个早晨起床冲咖啡的Vlog。"},
        {"role": "assistant", "content": "{'key': 'G Major', 'mode': 'major', 'tempo': '90bpm', 'instruments': 'acoustic guitar, light piano, snaps, soft shakers', 'genre': 'acoustic pop', 'feel': 'calm, positive, morning, gentle'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "科技产品评测视频的BGM。"},
        {"role": "assistant", "content": "{'key': 'G minor', 'mode': 'dorian', 'tempo': '110bpm', 'instruments': 'synth plucks, electronic drum beat, clean bassline', 'genre': 'electronic', 'feel': 'modern, neutral, focused, tech'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "美食Vlog，展示食物特写。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'major', 'tempo': '100bpm', 'instruments': 'upright bass, jazz piano, brushed drums, light saxophone', 'genre': 'jazz', 'feel': 'playful, happy, delicious'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个在东京旅行的vlog，很多城市夜景。"},
        {"role": "assistant", "content": "{'key': 'A minor', 'mode': 'minor', 'tempo': '85bpm', 'instruments': 'lo-fi electric piano, muffled drum beat, synth pad, vinyl crackle', 'genre': 'lo-fi hip hop', 'feel': 'urban, chill, nighttime, stylish'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "健身房锻炼的vlog，需要有活力的。"},
        {"role": "assistant", "content": "{'key': 'E minor', 'mode': 'minor', 'tempo': '125bpm', 'instruments': 'synth bass, strong electronic drums, aggressive synth lead', 'genre': 'EDM', 'feel': 'energetic, motivational, powerful'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要一段 sponsor segment（广告时段）的音乐。"},
        {"role": "assistant", "content": "{'key': 'F Major', 'mode': 'major', 'tempo': '115bpm', 'instruments': 'clean electric guitar, simple bass, light drums, claps', 'genre': 'corporate pop', 'feel': 'positive, friendly, engaging, neutral'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "学习vlog，用作背景音，不能太吵。"},
        {"role": "assistant", "content": "{'key': 'C minor', 'mode': 'minor', 'tempo': '80bpm', 'instruments': 'lo-fi piano, soft beat, ambient texture, no vocals', 'genre': 'lo-fi hip hop', 'feel': 'chill, study, focused, calm'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个DIY手工视频。"},
        {"role": "assistant", "content": "{'key': 'D Major', 'mode': 'major', 'tempo': '110bpm', 'instruments': 'ukulele, glockenspiel, acoustic guitar, light snaps', 'genre': 'acoustic', 'feel': 'creative, happy, light, crafty'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "无人机航拍，非常壮丽的自然风光。"},
        {"role": "assistant", "content": "{'key': 'G Major', 'mode': 'major', 'tempo': '90bpm', 'instruments': 'sweeping string orchestra, choir pad, gentle piano', 'genre': 'cinematic orchestral', 'feel': 'majestic, awe-inspiring, vast, beautiful'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "搞笑短片，需要滑稽一点的。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'major', 'tempo': '140bpm', 'instruments': 'tuba, clarinet, slide whistle, xylophone', 'genre': 'comedy', 'feel': 'funny, clumsy, goofy, wacky'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "在海边的vlog，很放松，有日落。"},
        {"role": "assistant", "content": "{'key': 'A Major', 'mode': 'major', 'tempo': '85bpm', 'instruments': 'soft acoustic guitar, steel drums, gentle wave sounds', 'genre': 'acoustic', 'feel': 'relaxed, peaceful, tropical, warm'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "展示一个app的教程视频。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'major', 'tempo': '110bpm', 'instruments': 'electric piano, minimal synth plucks, simple beat', 'genre': 'corporate minimal', 'feel': 'informative, clear, neutral, modern'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个关于宠物的可爱视频。"},
        {"role": "assistant", "content": "{'key': 'F Major', 'mode': 'major', 'tempo': '120bpm', 'instruments': 'ukulele, pizzicato strings, bells, cheerful whistling', 'genre': 'acoustic folk', 'feel': 'cute, happy, innocent, playful'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "在下雨天，室内看书的vlog。"},
        {"role": "assistant", "content": "{'key': 'F minor', 'mode': 'minor', 'tempo': '75bpm', 'instruments': 'gentle piano, soft string pad, rain sounds', 'genre': 'ambient', 'feel': 'cozy, melancholic, calm, introspective'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "化妆教程视频。"},
        {"role": "assistant", "content": "{'key': 'D-flat Major', 'mode': 'major', 'tempo': '100bpm', 'instruments': 'synth pop beat, clean electric guitar, female vocal pad', 'genre': 'pop', 'feel': 'chic, stylish, confident, modern'}"}
    ]},

    # --- Category 4: 播客/叙事 场景 (Podcast/Narrative) ---
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我是一个真实犯罪 (True Crime) 播客，正在讲述案情。"},
        {"role": "assistant", "content": "{'key': 'E minor', 'mode': 'minor', 'tempo': '80bpm', 'instruments': 'low synth drone, distant piano, sparse percussion', 'genre': 'dark ambient', 'feel': 'suspenseful, mysterious, dark, investigative'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "历史纪录片的BGM，关于古代战争。"},
        {"role": "assistant", "content": "{'key': 'C minor', 'mode': 'minor', 'tempo': '100bpm', 'instruments': 'epic strings, war drums, low brass section, choir', 'genre': 'epic orchestral', 'feel': 'serious, dramatic, historic, tense'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个关于太空探索的科普播客。"},
        {"role": "assistant", "content": "{'key': 'A minor', 'mode': 'dorian', 'tempo': '90bpm', 'instruments': 'ambient synth pads, arpeggiated synth, subtle electronic beat', 'genre': 'ambient electronic', 'feel': 'wonder, curious, futuristic, vast'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "情感故事播客，正在讲述一个感人的重逢。"},
        {"role": "assistant", "content": "{'key': 'E-flat Major', 'mode': 'major', 'tempo': '70bpm', 'instruments': 'solo piano, high strings crescendo, cello', 'genre': 'cinematic', 'feel': 'touching, emotional, heartwarming, hopeful'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "商业分析播客的片头。"},
        {"role": "assistant", "content": "{'key': 'G Major', 'mode': 'major', 'tempo': '120bpm', 'instruments': 'upbeat piano, driving bassline, clean drums, synth plucks', 'genre': 'corporate pop', 'feel': 'professional, energetic, smart, confident'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个关于哲学的深度对话播客。"},
        {"role": "assistant", "content": "{'key': 'D minor', 'mode': 'minor', 'tempo': '70bpm', 'instruments': 'ambient guitar, synth pad, no drums', 'genre': 'ambient', 'feel': 'introspective, thoughtful, calm, deep'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "调查记者播客，揭露黑幕。"},
        {"role": "assistant", "content": "{'key': 'F# minor', 'mode': 'minor', 'tempo': '110bpm', 'instruments': 'tense string ostinato, low piano notes, ticking clock', 'genre': 'suspense', 'feel': 'investigative, serious, tense, uncovering'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "晚间助眠播客，读诗。"},
        {"role": "assistant", "content": "{'key': 'D-flat Major', 'mode': 'major', 'tempo': '50bpm', 'instruments': 'very soft synth pad, gentle ambient textures, no beat', 'genre': 'ambient', 'feel': 'sleepy, calm, peaceful, ethereal'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "儿童故事播客。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'major', 'tempo': '110bpm', 'instruments': 'glockenspiel, pizzicato strings, flute, light percussion', 'genre': 'childrens', 'feel': 'magical, happy, whimsical, innocent'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个关于超自然现象的播客。"},
        {"role": "assistant", "content": "{'key': 'A minor', 'mode': 'phrygian', 'tempo': '80bpm', 'instruments': 'theremin, eerie synth pad, reversed piano, choir pad', 'genre': 'dark ambient', 'feel': 'spooky, mysterious, supernatural, strange'}"}
    ]},

    # --- Category 5: 动态/高级/风格化 请求 (Dynamic/Advanced) ---
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要一段音乐，从平缓寂寥开始。"},
        {"role": "assistant", "content": "{'key': 'E minor', 'mode': 'minor', 'tempo': '60bpm', 'instruments': 'ambient pad, sparse piano notes', 'genre': 'ambient', 'feel': 'calm, lonely, spacious'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要音乐在28秒后转为紧张暧昧。"},
        {"role": "assistant", "content": "{'key': 'E minor', 'mode': 'minor', 'tempo': '120bpm', 'instruments': 'pizzicato strings, low synth pulse, subtle hi-hats', 'genre': 'suspense', 'feel': 'tense, ambiguous, seductive'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "给我来点赛博朋克风格的，适合城市夜景。"},
        {"role": "assistant", "content": "{'key': 'F# minor', 'mode': 'minor', 'tempo': '140bpm', 'instruments': 'heavy synth bass, driving electronic drums, neon synth lead', 'genre': 'synthwave', 'feel': 'cyberpunk, futuristic, dark, energetic'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我想要一段中世纪幻想风格的音乐，像在酒馆里。"},
        {"role": "assistant", "content": "{'key': 'D minor', 'mode': 'dorian', 'tempo': '110bpm', 'instruments': 'lute, acoustic guitar, fiddle, hand drum', 'genre': 'fantasy folk', 'feel': 'medieval, tavern, cheerful, rustic'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个悬疑片的开头，主角登场。"},
        {"role": "assistant", "content": "{'key': 'A minor', 'mode': 'minor', 'tempo': '90bpm', 'instruments': 'low piano motif, tense string pad, subtle percussion', 'genre': 'cinematic suspense', 'feel': 'mysterious, introductory, tense'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要一个情绪的转折点，从悲伤转为坚定。"},
        {"role": "assistant", "content": "{'key': 'C minor', 'mode': 'minor', 'tempo': '100bpm', 'instruments': 'piano, full string section, subtle timpani roll', 'genre': 'cinematic', 'feel': 'transitioning, resolving, determined'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "8-bit游戏风格，欢快的。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'major', 'tempo': '130bpm', 'instruments': 'chiptune synth lead, 8-bit drums, square wave bass', 'genre': 'chiptune', 'feel': 'retro, happy, video game, nostalgic'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一段非常简约的音乐，适合冥想。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'lydian', 'tempo': '50bpm', 'instruments': 'ambient synth pad, crystal bowls, no beat', 'genre': 'meditation', 'feel': 'minimalist, peaceful, ethereal, healing'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "西部片风格，决斗前的对峙。"},
        {"role": "assistant", "content": "{'key': 'A minor', 'mode': 'phrygian', 'tempo': '80bpm', 'instruments': 'western guitar, whistling, distant bells, snare drum roll', 'genre': 'western', 'feel': 'tense, standoff, classic, dramatic'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个盛大的颁奖典礼BGM。"},
        {"role": "assistant", "content": "{'key': 'D Major', 'mode': 'major', 'tempo': '100bpm', 'instruments': 'full orchestra, brass fanfare, timpani, choir', 'genre': 'orchestral', 'feel': 'grand, celebratory, prestigious, uplifting'}"}
    ]},

    # --- Category 6: 更多情绪/场景组合 (原100条) ---
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "忧郁的，但不是完全绝望。"},
        {"role": "assistant", "content": "{'key': 'F minor', 'mode': 'minor', 'tempo': '70bpm', 'instruments': 'solo viola, rainy day piano, soft string pad', 'genre': 'cinematic', 'feel': 'melancholy, introspective, gray'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "充满活力的，像早晨的太阳。"},
        {"role": "assistant", "content": "{'key': 'E Major', 'mode': 'major', 'tempo': '115bpm', 'instruments': 'acoustic guitar, bright piano, hand claps, light bass', 'genre': 'acoustic pop', 'feel': 'energetic, bright, positive, fresh'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要一点恐怖的氛围。"},
        {"role": "assistant", "content": "{'key': 'D# minor', 'mode': 'minor', 'tempo': '60bpm', 'instruments': 'dissonant string cluster, low piano rumble, waterphone', 'genre': 'horror ambient', 'feel': 'scary, dark, unsettling, suspenseful'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个孩子在玩耍的场景。"},
        {"role": "assistant", "content": "{'key': 'G Major', 'mode': 'major', 'tempo': '125bpm', 'instruments': 'ukulele, xylophone, light percussion, whistling', 'genre': 'childrens', 'feel': 'innocent, playful, happy, light'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "放松的爵士乐，适合咖啡馆。"},
        {"role": "assistant", "content": "{'key': 'B-flat Major', 'mode': 'major', 'tempo': '90bpm', 'instruments': 'jazz piano trio (piano, upright bass, brushed drums)', 'genre': 'jazz', 'feel': 'relaxing, cozy, sophisticated, coffee shop'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个宏大的史诗战斗场面。"},
        {"role": "assistant", "content": "{'key': 'C minor', 'mode': 'minor', 'tempo': '150bpm', 'instruments': 'full orchestra, aggressive choir, war drums, brass section', 'genre': 'epic orchestral', 'feel': 'epic, battle, intense, dramatic'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "复古的，80年代的感觉。"},
        {"role": "assistant", "content": "{'key': 'E minor', 'mode': 'minor', 'tempo': '120bpm', 'instruments': 'analog synth lead, drum machine (linndrum), synth bass, reverb guitar', 'genre': 'synthwave', 'feel': 'retro, 80s, nostalgic, cool'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个悬念揭晓的时刻。"},
        {"role": "assistant", "content": "{'key': 'G minor', 'mode': 'minor', 'tempo': '100bpm', 'instruments': 'string staccato, timpani hit, cymbal swell, silence', 'genre': 'cinematic suspense', 'feel': 'reveal, dramatic, intense, climax'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "梦幻的，飘渺的感觉。"},
        {"role": "assistant", "content": "{'key': 'F Major', 'mode': 'lydian', 'tempo': '70bpm', 'instruments': 'harp arpeggios, soft string pads, gentle choir, celesta', 'genre': 'ambient', 'feel': 'dreamy, ethereal, magical, floating'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个很酷的侦探主题。"},
        {"role": "assistant", "content": "{'key': 'A minor', 'mode': 'minor', 'tempo': '95bpm', 'instruments': 'walking bassline, ride cymbal, moody saxophone, jazz piano', 'genre': 'jazz noir', 'feel': 'cool, mysterious, detective, smoky'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要一段适合做新闻开场的音乐。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'major', 'tempo': '130bpm', 'instruments': 'orchestral hits, synth melody, driving percussion, brass fanfare', 'genre': 'news theme', 'feel': 'authoritative, energetic, informative, modern'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个非常鼓舞人心的演讲BGM。"},
        {"role": "assistant", "content": "{'key': 'G Major', 'mode': 'major', 'tempo': '110bpm', 'instruments': 'piano chords, uplifting strings, gentle build-up, clean percussion', 'genre': 'cinematic pop', 'feel': 'inspirational, motivational, hopeful, positive'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "失败的，沮丧的。"},
        {"role": "assistant", "content": "{'key': 'B minor', 'mode': 'minor', 'tempo': '70bpm', 'instruments': 'dissonant piano, low strings, cello solo', 'genre': 'cinematic', 'feel': 'defeated, frustrated, somber, down'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "给我来点日系Vlog的感觉。"},
        {"role": "assistant", "content": "{'key': 'F Major', 'mode': 'major', 'tempo': '95bpm', 'instruments': 'clean electric guitar, upright piano, simple bass, soft drum beat', 'genre': 'j-pop acoustic', 'feel': 'light, everyday, cozy, simple'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个产品发布会的开场，要震撼。"},
        {"role": "assistant", "content": "{'key': 'C minor', 'mode': 'minor', 'tempo': '120bpm', 'instruments': 'heavy synth bass, electronic drums, orchestral strings, glitch effects', 'genre': 'electronic orchestral', 'feel': 'epic, modern, powerful, tech, launch'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我正在剪一个关于'遗憾'的短片。"},
        {"role": "assistant", "content": "{'key': 'G minor', 'mode': 'minor', 'tempo': '65bpm', 'instruments': 'solo piano with heavy reverb, distant string pad', 'genre': 'cinematic ambient', 'feel': 'regretful, nostalgic, melancholic, empty'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "开车在路上的感觉，很自由。"},
        {"role": "assistant", "content": "{'key': 'E Major', 'mode': 'major', 'tempo': '120bpm', 'instruments': 'acoustic guitar, driving drum beat, bass guitar, harmonica', 'genre': 'folk rock', 'feel': 'free, driving, adventurous, open road'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个戏剧性的反转。"},
        {"role": "assistant", "content": "{'key': 'F minor', 'mode': 'minor', 'tempo': '130bpm', 'instruments': 'sudden orchestral hit, fast strings, rising pitch, timpani roll', 'genre': 'cinematic suspense', 'feel': 'dramatic, plot twist, shocking, intense'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "放松的雷鬼音乐。"},
        {"role": "assistant", "content": "{'key': 'A Major', 'mode': 'major', 'tempo': '80bpm', 'instruments': 'reggae guitar skank, bass guitar, one-drop drum beat, organ', 'genre': 'reggae', 'feel': 'relaxed, chill, groovy, positive'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个关于回忆过去的蒙太奇。"},
        {"role": "assistant", "content": "{'key': 'D Major', 'mode': 'major', 'tempo': '80bpm', 'instruments': 'gentle piano, warm string orchestra, acoustic guitar', 'genre': 'cinematic', 'feel': 'nostalgic, reflective, warm, montage'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "在森林里，很神秘。"},
        {"role": "assistant", "content": "{'key': 'E minor', 'mode': 'dorian', 'tempo': '75bpm', 'instruments': 'flute, harp, ambient forest sounds, soft string pad', 'genre': 'ambient', 'feel': 'mysterious, magical, fantasy, forest'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要一个很“潮”的嘻哈BGM。"},
        {"role": "assistant", "content": "{'key': 'G minor', 'mode': 'minor', 'tempo': '140bpm', 'instruments': '808 bass, trap hi-hats, synth melody, vocal chop', 'genre': 'trap', 'feel': 'modern, cool, trendy, energetic'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "紧张的追逐戏。"},
        {"role": "assistant", "content": "{'key': 'D minor', 'mode': 'minor', 'tempo': '160bpm', 'instruments': 'fast string ostinato, aggressive percussion, low brass stabs', 'genre': 'action orchestral', 'feel': 'chase, intense, urgent, fast'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个感人的独白。"},
        {"role": "assistant", "content": "{'key': 'F Major', 'mode': 'major', 'tempo': '60bpm', 'instruments': 'solo piano, very light string underscore', 'genre': 'cinematic', 'feel': 'emotional, monologue, touching, minimal'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个温馨的家庭聚会。"},
        {"role": "assistant", "content": "{'key': 'G Major', 'mode': 'major', 'tempo': '100bpm', 'instruments': 'acoustic guitar, gentle piano, brushed drums, light bass', 'genre': 'acoustic', 'feel': 'warm, happy, family, cozy'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要一点禅意的音乐。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'pentatonic', 'tempo': '60bpm', 'instruments': 'koto, shakuhachi flute, ambient pad', 'genre': 'world', 'feel': 'zen, meditative, peaceful, japanese'}"}
    ]},

    # --- Category 7: 更多风格和复杂指令 ---
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要一段福音音乐 (Gospel)。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'major', 'tempo': '120bpm', 'instruments': 'hammond organ, gospel choir, piano, drums', 'genre': 'gospel', 'feel': 'joyful, uplifting, spiritual'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "法国电影的感觉，有点手风琴。"},
        {"role": "assistant", "content": "{'key': 'A minor', 'mode': 'minor', 'tempo': '110bpm', 'instruments': 'accordion, acoustic guitar, upright bass', 'genre': 'world', 'feel': 'romantic, nostalgic, parisian'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "阴郁的，哥特式的。"},
        {"role": "assistant", "content": "{'key': 'D minor', 'mode': 'minor', 'tempo': '70bpm', 'instruments': 'pipe organ, dark choir, harpsichord, slow strings', 'genre': 'classical', 'feel': 'dark, gothic, ominous, melancholic'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "律动感很强的Funk音乐。"},
        {"role": "assistant", "content": "{'key': 'E minor', 'mode': 'dorian', 'tempo': '115bpm', 'instruments': 'funky bassline, clean electric guitar riff, drums, brass hits', 'genre': 'funk', 'feel': 'groovy, upbeat, energetic'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一段蓝调 (Blues)。"},
        {"role": "assistant", "content": "{'key': 'A Major', 'mode': 'mixolydian', 'tempo': '100bpm', 'instruments': 'blues guitar, harmonica, upright bass, brushed drums', 'genre': 'blues', 'feel': 'soulful, gritty, melancholic'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要一段安静的背景音，没有旋律，只有氛围。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'major', 'tempo': '60bpm', 'instruments': 'ambient synth pad, wind texture, no percussion, no melody', 'genre': 'ambient', 'feel': 'atmospheric, calm, minimal'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "公司年会开场，积极向上的。"},
        {"role": "assistant", "content": "{'key': 'F Major', 'mode': 'major', 'tempo': '125bpm', 'instruments': 'upbeat pop drums, piano chords, clean electric guitar, synth lead', 'genre': 'corporate pop', 'feel': 'positive, energetic, celebratory, professional'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "视频结尾，感谢观众。"},
        {"role": "assistant", "content": "{'key': 'G Major', 'mode': 'major', 'tempo': '90bpm', 'instruments': 'gentle piano, light strings, acoustic guitar', 'genre': 'acoustic', 'feel': 'grateful, warm, closing, positive'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "激烈的辩论场景。"},
        {"role": "assistant", "content": "{'key': 'F minor', 'mode': 'minor', 'tempo': '130bpm', 'instruments': 'fast string staccato, driving percussion, low piano', 'genre': 'cinematic suspense', 'feel': 'tense, argumentative, urgent'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要一段“Lo-Fi”，但是用吉他为主。"},
        {"role": "assistant", "content": "{'key': 'A minor', 'mode': 'minor', 'tempo': '80bpm', 'instruments': 'lo-fi acoustic guitar loop, muffled beat, vinyl crackle, warm bass', 'genre': 'lo-fi hip hop', 'feel': 'chill, nostalgic, warm, study'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "在沙漠中行走，孤独炎热。"},
        {"role": "assistant", "content": "{'key': 'E minor', 'mode': 'phrygian', 'tempo': '70bpm', 'instruments': 'ethnic flute, sparse percussion, ambient pad, acoustic guitar', 'genre': 'world', 'feel': 'hot, desolate, lonely, atmospheric'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "夏日海滩派对。"},
        {"role": "assistant", "content": "{'key': 'G Major', 'mode': 'major', 'tempo': '110bpm', 'instruments': 'steel drums, ukulele, pop drums, bass guitar', 'genre': 'pop', 'feel': 'happy, summer, party, upbeat'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个非常古怪的、实验性的BGM。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'atonal', 'tempo': '100bpm', 'instruments': 'synth blips, random percussion, reversed sounds, theremin', 'genre': 'experimental', 'feel': 'quirky, strange, unpredictable'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "主角失败了，但正准备重新站起来。"},
        {"role": "assistant", "content": "{'key': 'C minor', 'mode': 'minor', 'tempo': '80bpm', 'instruments': 'low piano, rising string section, subtle crescendo', 'genre': 'cinematic', 'feel': 'defeated but rising, determined, comeback'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "给我来点卧室流行 (Bedroom Pop)。"},
        {"role": "assistant", "content": "{'key': 'F Major', 'mode': 'major', 'tempo': '95bpm', 'instruments': 'dreamy electric guitar, soft drum machine, synth pad, light bass', 'genre': 'bedroom pop', 'feel': 'dreamy, nostalgic, chill, indie'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个快速剪辑的vlog蒙太奇。"},
        {"role": "assistant", "content": "{'key': 'B minor', 'mode': 'minor', 'tempo': '128bpm', 'instruments': 'upbeat electronic beat, synth plucks, driving bass', 'genre': 'electronic', 'feel': 'fast, energetic, dynamic, montage'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个关于大自然的纪录片。"},
        {"role": "assistant", "content": "{'key': 'D Major', 'mode': 'major', 'tempo': '90bpm', 'instruments': 'full orchestra, woodwinds, majestic strings, harp', 'genre': 'orchestral', 'feel': 'majestic, beautiful, nature, wondrous'}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要一段时长20秒的，平静的冥想音乐。"},
        {"role": "assistant", "content": "{'key': 'F Major', 'mode': 'lydian', 'tempo': '55bpm', 'instruments': 'ambient synth pad, crystal bowls', 'genre': 'meditation', 'feel': 'calm, peaceful, serene', 'duration_sec': 20}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个30秒的广告，产品是咖啡，要温暖的。"},
        {"role": "assistant", "content": "{'key': 'G Major', 'mode': 'major', 'tempo': '90bpm', 'instruments': 'acoustic guitar, gentle piano, light snaps', 'genre': 'acoustic', 'feel': 'warm, cozy, inviting', 'duration_sec': 30}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要一段15秒的紧张音效，用于短视频。"},
        {"role": "assistant", "content": "{'key': 'D minor', 'mode': 'minor', 'tempo': '140bpm', 'instruments': 'fast string staccato, low brass hit, ticking clock', 'genre': 'suspense', 'feel': 'tense, urgent', 'duration_sec': 15}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "45秒的vlog开场，日系，积极。"},
        {"role": "assistant", "content": "{'key': 'F Major', 'mode': 'major', 'tempo': '110bpm', 'instruments': 'upright piano, clean electric guitar, simple bass', 'genre': 'j-pop acoustic', 'feel': 'positive, light, everyday', 'duration_sec': 45}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "时长1分钟，史诗感的。"},
        {"role": "assistant", "content": "{'key': 'D Major', 'mode': 'major', 'tempo': '140bpm', 'instruments': 'full orchestra, epic choir, taiko drums', 'genre': 'epic orchestral', 'feel': 'grand, heroic, majestic', 'duration_sec': 60}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要一段10秒的，搞笑的音效，像bloopers。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'major', 'tempo': '130bpm', 'instruments': 'tuba, slide whistle, xylophone', 'genre': 'comedy', 'feel': 'funny, clumsy, goofy', 'duration_sec': 10}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一段2分钟的lo-fi，用来学习。"},
        {"role": "assistant", "content": "{'key': 'C minor', 'mode': 'minor', 'tempo': '80bpm', 'instruments': 'lo-fi piano, soft beat, ambient texture', 'genre': 'lo-fi hip hop', 'feel': 'chill, study, focused, calm', 'duration_sec': 120}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "30秒，赛博朋克风格，用于产品展示。"},
        {"role": "assistant", "content": "{'key': 'F# minor', 'mode': 'minor', 'tempo': '140bpm', 'instruments': 'heavy synth bass, driving electronic drums, neon synth lead', 'genre': 'synthwave', 'feel': 'cyberpunk, futuristic, dark, energetic', 'duration_sec': 30}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一个非常缓慢的，1分钟的悲伤钢琴独奏。"},
        {"role": "assistant", "content": "{'key': 'A minor', 'mode': 'minor', 'tempo': '60bpm', 'instruments': 'solo piano', 'genre': 'classical', 'feel': 'somber, melancholic, reflective', 'duration_sec': 60}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "15秒，恐怖音效，要一个jump scare。"},
        {"role": "assistant", "content": "{'key': 'D# minor', 'mode': 'minor', 'tempo': '80bpm', 'instruments': 'dissonant string cluster, sudden loud orchestral hit', 'genre': 'horror ambient', 'feel': 'scary, jump scare, unsettling', 'duration_sec': 15}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要40秒的浪漫音乐，适合婚礼视频。"},
        {"role": "assistant", "content": "{'key': 'E-flat Major', 'mode': 'major', 'tempo': '80bpm', 'instruments': 'romantic piano, warm strings', 'genre': 'cinematic', 'feel': 'romantic, intimate, warm, loving', 'duration_sec': 40}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "时长10秒，颁奖典礼的铜管乐。"},
        {"role": "assistant", "content": "{'key': 'B-flat Major', 'mode': 'major', 'tempo': '120bpm', 'instruments': 'brass fanfare, timpani roll', 'genre': 'orchestral', 'feel': 'victorious, triumphant, grand', 'duration_sec': 10}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一段90秒的，非常自信的摇滚BGM。"},
        {"role": "assistant", "content": "{'key': 'E minor', 'mode': 'minor', 'tempo': '120bpm', 'instruments': 'electric guitar riff, solid drum beat, synth bass', 'genre': 'rock', 'feel': 'cool, confident, edgy', 'duration_sec': 90}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "时长30秒，日系，但有点悲伤的。"},
        {"role": "assistant", "content": "{'key': 'G minor', 'mode': 'minor', 'tempo': '85bpm', 'instruments': 'piano melody, light string pad', 'genre': 'j-pop ballad', 'feel': 'melancholic, bittersweet, gentle', 'duration_sec': 30}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要一段20秒的，非常简约的音乐，适合冥想。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'lydian', 'tempo': '50bpm', 'instruments': 'ambient synth pad, crystal bowls, no beat', 'genre': 'meditation', 'feel': 'minimalist, peaceful, ethereal, healing', 'duration_sec': 20}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "30秒，适合播客片头，关于科技的。"},
        {"role": "assistant", "content": "{'key': 'G minor', 'mode': 'dorian', 'tempo': '110bpm', 'instruments': 'synth plucks, electronic drum beat, clean bassline', 'genre': 'electronic', 'feel': 'modern, neutral, focused, tech', 'duration_sec': 30}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "时长1分钟，怀旧的，快乐的童年回忆。"},
        {"role": "assistant", "content": "{'key': 'F Major', 'mode': 'major', 'tempo': '100bpm', 'instruments': 'music box, glockenspiel, light acoustic guitar', 'genre': 'acoustic', 'feel': 'nostalgic, happy, innocent', 'duration_sec': 60}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "15秒，非常宏大的史诗战斗。"},
        {"role": "assistant", "content": "{'key': 'C minor', 'mode': 'minor', 'tempo': '150bpm', 'instruments': 'full orchestra, aggressive choir, war drums', 'genre': 'epic orchestral', 'feel': 'epic, battle, intense, dramatic', 'duration_sec': 15}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一段25秒的，俏皮的、好玩的音乐。"},
        {"role": "assistant", "content": "{'key': 'G Major', 'mode': 'major', 'tempo': '130bpm', 'instruments': 'pizzicato strings, xylophone, clarinet', 'genre': 'comedy', 'feel': 'playful, quirky, fun', 'duration_sec': 25}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "时长1分钟，神圣庄严的。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'major', 'tempo': '60bpm', 'instruments': 'pipe organ, church choir pad, tubular bells', 'genre': 'classical', 'feel': 'sacred, solemn, majestic', 'duration_sec': 60}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "我需要30秒的，非常酷的侦探主题。"},
        {"role": "assistant", "content": "{'key': 'A minor', 'mode': 'minor', 'tempo': '95bpm', 'instruments': 'walking bassline, ride cymbal, moody saxophone', 'genre': 'jazz noir', 'feel': 'cool, mysterious, detective, smoky', 'duration_sec': 30}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "一段40秒的，非常鼓舞人心的音乐。"},
        {"role": "assistant", "content": "{'key': 'G Major', 'mode': 'major', 'tempo': '110bpm', 'instruments': 'piano chords, uplifting strings, gentle build-up', 'genre': 'cinematic pop', 'feel': 'inspirational, motivational, hopeful, positive', 'duration_sec': 40}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "15秒，产品发布会的震撼开场。"},
        {"role": "assistant", "content": "{'key': 'C minor', 'mode': 'minor', 'tempo': '120bpm', 'instruments': 'heavy synth bass, electronic drums, orchestral strings', 'genre': 'electronic orchestral', 'feel': 'epic, modern, powerful, tech, launch', 'duration_sec': 15}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "时长1分钟，关于遗憾的短片。"},
        {"role": "assistant", "content": "{'key': 'G minor', 'mode': 'minor', 'tempo': '65bpm', 'instruments': 'solo piano with heavy reverb, distant string pad', 'genre': 'cinematic ambient', 'feel': 'regretful, nostalgic, melancholic, empty', 'duration_sec': 60}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "30秒，开车在路上，很自由。"},
        {"role": "assistant", "content": "{'key': 'E Major', 'mode': 'major', 'tempo': '120bpm', 'instruments': 'acoustic guitar, driving drum beat, bass guitar', 'genre': 'folk rock', 'feel': 'free, driving, adventurous, open road', 'duration_sec': 30}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "10秒，戏剧性反转的音效。"},
        {"role": "assistant", "content": "{'key': 'F minor', 'mode': 'minor', 'tempo': '130bpm', 'instruments': 'sudden orchestral hit, fast strings, rising pitch', 'genre': 'cinematic suspense', 'feel': 'dramatic, plot twist, shocking, intense', 'duration_sec': 10}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "1分钟，放松的雷鬼音乐。"},
        {"role": "assistant", "content": "{'key': 'A Major', 'mode': 'major', 'tempo': '80bpm', 'instruments': 'reggae guitar skank, bass guitar, one-drop drum beat', 'genre': 'reggae', 'feel': 'relaxed, chill, groovy, positive', 'duration_sec': 60}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "30秒，回忆蒙太奇。"},
        {"role": "assistant", "content": "{'key': 'D Major', 'mode': 'major', 'tempo': '80bpm', 'instruments': 'gentle piano, warm string orchestra', 'genre': 'cinematic', 'feel': 'nostalgic, reflective, warm, montage', 'duration_sec': 30}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "20秒，神秘的森林。"},
        {"role": "assistant", "content": "{'key': 'E minor', 'mode': 'dorian', 'tempo': '75bpm', 'instruments': 'flute, harp, ambient forest sounds', 'genre': 'ambient', 'feel': 'mysterious, magical, fantasy, forest', 'duration_sec': 20}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "30秒，很潮的嘻哈BGM。"},
        {"role": "assistant", "content": "{'key': 'G minor', 'mode': 'minor', 'tempo': '140bpm', 'instruments': '808 bass, trap hi-hats, synth melody', 'genre': 'trap', 'feel': 'modern, cool, trendy, energetic', 'duration_sec': 30}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "45秒，紧张的追逐戏。"},
        {"role": "assistant", "content": "{'key': 'D minor', 'mode': 'minor', 'tempo': '160bpm', 'instruments': 'fast string ostinato, aggressive percussion, low brass stabs', 'genre': 'action orchestral', 'feel': 'chase, intense, urgent, fast', 'duration_sec': 45}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "1分钟，感人的独白。"},
        {"role": "assistant", "content": "{'key': 'F Major', 'mode': 'major', 'tempo': '60bpm', 'instruments': 'solo piano, very light string underscore', 'genre': 'cinematic', 'feel': 'emotional, monologue, touching, minimal', 'duration_sec': 60}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "30秒，温馨的家庭聚会。"},
        {"role": "assistant", "content": "{'key': 'G Major', 'mode': 'major', 'tempo': '100bpm', 'instruments': 'acoustic guitar, gentle piano, brushed drums', 'genre': 'acoustic', 'feel': 'warm, happy, family, cozy', 'duration_sec': 30}"}
    ]},
    {"messages": [
        {"role": "system", "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。"},
        {"role": "user", "content": "45秒，禅意的音乐。"},
        {"role": "assistant", "content": "{'key': 'C Major', 'mode': 'pentatonic', 'tempo': '60bpm', 'instruments': 'koto, shakuhachi flute, ambient pad', 'genre': 'world', 'feel': 'zen, meditative, peaceful, japanese', 'duration_sec': 45}"}
    ]}
]

# --- Dataset Splitting Strategy ---
# To monitor the model's ability to generalize (and detect overfitting),
# we must separate a portion of data that the model will never see during training.

# 1. Convert Python list to Hugging Face Dataset object
full_dataset = Dataset.from_list(sft_data)

# 2. Perform Train/Test Split
# Allocating 90% for Training and 10% for Evaluation/Testing.
# Seed is set to 42 for reproducibility.
split_dataset = full_dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

# 3. Validation (Optional export logic omitted)
# Using in-memory datasets for Colab efficiency.

print(f"--- Dataset Preparation Complete ---")
print(f"Total Examples: {len(full_dataset)}")
print(f"Training Set:   {len(train_dataset)}")
print(f"Evaluation Set: {len(eval_dataset)}")
print("\n--- Evaluation Sample (Unseen by Model) ---")
print(json.dumps(eval_dataset[0], indent=2, ensure_ascii=False))

--- 数据集准备完成 (已修复所有错误) ---
总数据: 137 条
训练集 (Train): 123 条
评估集 (Eval): 14 条

--- 评估集 (Eval) 示例 (模型没见过这些数据) ---
{
  "messages": [
    {
      "content": "你是一个AI音乐导演, 负责将用户的情绪描述翻译成详细的音乐生成参数。",
      "role": "system"
    },
    {
      "content": "阴郁的，哥特式的。",
      "role": "user"
    },
    {
      "content": "{'key': 'D minor', 'mode': 'minor', 'tempo': '70bpm', 'instruments': 'pipe organ, dark choir, harpsichord, slow strings', 'genre': 'classical', 'feel': 'dark, gothic, ominous, melancholic'}",
      "role": "assistant"
    }
  ]
}


In [ ]:
"""
Cell 4: Main Training Script with Visualization Support

This script initializes the Qwen model with 4-bit quantization, configures the 
LoRA (Low-Rank Adaptation) adapter, and executes the training loop using the 
SFTTrainer. It is configured to report metrics to Weights & Biases (W&B) 
for real-time visualization of loss curves.
"""

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer

# --- 1. Model Definition ---
model_name = "Qwen/Qwen1.5-4B-Chat"

# --- 2. BitsAndBytes (4-bit Quantization) Configuration ---
# Optimization for memory efficiency on consumer GPUs
compute_dtype = getattr(torch, "float16")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

# --- 3. Load Model and Tokenizer ---
print("--- Loading Model... ---")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print("--- Loading Tokenizer... ---")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# --- 4. Dataset Integration ---
# Importing the 'train_dataset' and 'eval_dataset' objects directly from memory (Cell 3)
print("--- Integrating Datasets from Cell 3... ---")
print(f"Training Samples:   {len(train_dataset)}")
print(f"Evaluation Samples: {len(eval_dataset)}")

# ----------------------------------------------------------------------
# EXPERIMENT CONFIGURATION
# ----------------------------------------------------------------------
# Adjust these parameters to simulate different training outcomes.
#
# Option A: Underfitting (Rank=8, Epochs=3)
# Option B: Best Fit     (Rank=32, Epochs=3 - Early Stopping)
# Option C: Overfitting  (Rank=32, Epochs=25)

# --- Current Configuration: Option B (Best Fit) ---
experiment_name = "Experiment-2-Rank-32-Best-Fit"
lora_r = 32          # Rank dimension (Capacity of the adapter)
lora_alpha = 64      # Scaling factor (Strength of the update)
num_epochs = 3       # Number of passes through the dataset
# ----------------------------------------------------------------------

# --- 5. LoRA Adapter Configuration ---
peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

# --- 6. Training Arguments ---
training_arguments = TrainingArguments(
    output_dir=f"./results-{experiment_name}", # Separate directory for each experiment
    num_train_epochs=num_epochs,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    logging_steps=5,                  # Log training loss every 5 steps
    save_strategy="no",               # Checkpoints disabled to save disk space
    fp16=True,
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    # --- Visualization Settings (W&B) ---
    evaluation_strategy="steps",      # Enable evaluation during training
    eval_steps=10,                    # Run evaluation against the test set every 10 steps
    report_to="wandb",                # Log metrics to Weights & Biases
    run_name=experiment_name          # Experiment identifier in the W&B dashboard
)

# --- 7. Initialize SFTTrainer ---
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=False,
)

# --- 8. Execute Training ---
print(f"\n--- Starting Training: {experiment_name} ---")
trainer.train()
print(f"--- Training Complete: {experiment_name} ---")

# Note: The 'model' object is now updated with the latest trained weights.

--- 正在加载模型... ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)



--- 开始训练: Experiment-2-Rank-32-Target ---


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss,Validation Loss
10,2.678200,1.765938
20,0.959000,0.798031
30,0.825700,0.672161
40,0.595200,0.665788
50,0.546600,0.636302
60,0.527600,0.624151
70,0.338400,0.682530
80,0.293500,0.675102
90,0.331100,0.658351
100,0.189500,0.732927


--- 训练完成: Experiment-2-Rank-32-Target ---


In [ ]:
"""
Module: Professional Test Set & Quantitative Evaluation

This module provides the infrastructure for running inference on the fine-tuned model
and evaluating its output quality. It includes:
1. Device and model preparation.
2. Prompt construction using chat templates.
3. A robust wrapper for `model.generate`.
4. Domain-specific utility functions for music theory evaluation.

Pre-requisites: 'trainer' and 'tokenizer' objects must exist in the environment.
"""

import json, re, difflib, time
from typing import Optional
import torch
import pandas as pd
import math

# -----------------------------
# 1. Device & Model Setup
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to the correct device if necessary
try:
    trainer.model.to(device)
except Exception:
    pass # Model might already be on the correct device or handled by Accelerator

# Set model to evaluation mode (disables dropout, etc.)
trainer.model.eval()

# Enable KV caching for faster inference
try:
    trainer.model.config.use_cache = True
except Exception:
    pass

# -----------------------------
# 2. Prompt Engineering
# -----------------------------
def build_prompt(system_prompt: str, user_prompt: str) -> str:
    """
    Constructs the full prompt string. 
    Prioritizes `tokenizer.apply_chat_template` if available for best model compatibility.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    
    # Attempt to use the tokenizer's built-in chat template (e.g., ChatML, Llama format)
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
            
    # Fallback to a standard format if templates fail
    return f"{system_prompt}\n\nUser: {user_prompt}\n\nAssistant:"

# -----------------------------
# 3. Inference Wrapper
# -----------------------------
def get_model_response(user_prompt: str,
                       system_prompt: str = "You are an AI Music Director. Translate the user's emotional description into detailed music generation parameters.",
                       max_new_tokens: int = 256,
                       do_sample: bool = True,
                       temperature: float = 0.01,
                       top_p: float = 0.9) -> Optional[str]:
    """
    Generates a response from the model.
    
    Args:
        user_prompt: The input instruction.
        system_prompt: The persona/context for the model.
        max_new_tokens: Maximum length of generated text.
        temperature: Controls randomness (low = deterministic).
        top_p: Nucleus sampling parameter.
        
    Returns:
        str: The raw text response from the assistant.
    """
    prompt = build_prompt(system_prompt, user_prompt)
    
    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs.get("attention_mask", None)
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)

    # Standard generation parameters
    gen_kwargs = dict(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    # Execute generation (handling version differences in transformers)
    try:
        with torch.no_grad():
            outputs = trainer.model.generate(**gen_kwargs)
    except TypeError:
        # Fallback for older versions that might reject **kwargs unpacking for some args
        with torch.no_grad():
            outputs = trainer.model.generate(
                input_ids, 
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                temperature=temperature,
                top_p=top_p,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True
            )

    if outputs is None:
        return None

    # Post-processing: Decode and extract the new content
    generated_ids = outputs[0]
    prompt_len = input_ids.shape[1]
    
    # If the output length is unexpectedly short, try full decode
    if generated_ids.shape[0] <= prompt_len:
        full_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        if full_text.startswith(prompt):
            return full_text[len(prompt):].strip()
        return full_text.strip()
    
    # Standard extraction: slice off the prompt tokens
    new_tokens = generated_ids[prompt_len:]
    assistant_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    # Final safety check if slicing returned empty string
    if assistant_text == "":
        full_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        if full_text.startswith(prompt):
            assistant_text = full_text[len(prompt):].strip()
        else:
            assistant_text = full_text.strip()
            
    return assistant_text

# -----------------------------
# 4. Evaluation Utilities (Music Expert Logic)
# -----------------------------

# Define sets of musical modes for theory validation checks
minor_modes = {"aeolian", "dorian", "phrygian", "locrian"}  # Examples of minor-quality modes

def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = s.lower()
    s = re.sub(r"\(.*?\)", "", s)
    s = re.sub(r"[^0-9a-zA-Z\u4e00-\u9fff,]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def jaccard_similarity(a: str, b: str) -> float:
    set_a = set([t for t in re.split(r'[,\s]+', a) if t])
    set_b = set([t for t in re.split(r'[,\s]+', b) if t])
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

def seq_ratio(a: str, b: str) -> float:
    return difflib.SequenceMatcher(None, a, b).ratio()

def is_minor_family(mode_str: str) -> bool:
    if not mode_str:
        return False
    mode = normalize_text(mode_str)
    if "minor" in mode:
        return True
    for m in minor_modes:
        if m in mode:
            return True
    return False

def match_value_score(expected: str, actual: str) -> float:
    exp = expected
    act = actual if actual is not None else ""
    exp_n = normalize_text(exp)
    act_n = normalize_text(act)

    # forbidden
    if exp_n.startswith("not "):
        forbid = exp_n[4:].strip()
        if forbid == "":
            return 1.0
        if forbid in act_n:
            return 0.0
        return 1.0

    # special-case mode minor/major
    if exp_n in {"minor", "major"}:
        if exp_n == "minor":
            return 1.0 if is_minor_family(act_n) else 0.0
        else:
            return 1.0 if "major" in act_n else 0.0

    # Jaccard + sequence ratio combination
    jacc = jaccard_similarity(exp_n, act_n)
    seq = seq_ratio(exp_n, act_n)
    score = 0.7 * jacc + 0.3 * seq

    # If expected is single token and included in actual, boost
    exp_tokens = [t for t in re.split(r'[,\s]+', exp_n) if t]
    if len(exp_tokens) == 1 and exp_tokens[0] in act_n:
        return max(score, 0.9)
    return score

def evaluate_response_improved(response_text, expected_keys=[], per_key_threshold=0.6, overall_threshold=0.7):
    result = {
        "json_valid": 0,
        "parsed_json": None,
        "per_key_scores": {},
        "keys_matched": 0,
        "cleaned_text": None
    }

    def clean_text(t):
        if t is None:
            return ""
        t = t.strip()
        m = re.search(r"```(?:json)?\s*(.*?)\s*```", t, flags=re.S|re.I)
        if m:
            t = m.group(1)
        m2 = re.search(r"(\{[\s\S]*\})", t)
        if m2:
            t = m2.group(1)
        if t.startswith("{") and '"' not in t and "'" in t:
            t = t.replace("':", '":').replace(":'", ':"').replace("'", '"')
        return t

    cleaned = clean_text(response_text)
    result["cleaned_text"] = cleaned
    try:
        parsed = json.loads(cleaned)
        result["json_valid"] = 1
        result["parsed_json"] = parsed
    except Exception:
        result["json_valid"] = 0
        return result

    per_scores = {}
    for (key, expected_value) in expected_keys:
        actual_value = parsed.get(key, "")
        if expected_value is None:
            per_scores[key] = 1.0 if key in parsed else 0.0
            continue
        if isinstance(expected_value, (list, tuple)):
            scores = [match_value_score(ev, str(actual_value)) for ev in expected_value]
            per_scores[key] = max(scores) if scores else 0.0
        else:
            per_scores[key] = match_value_score(str(expected_value), str(actual_value))
    result["per_key_scores"] = per_scores

    avg_score = sum(per_scores.values()) / (len(per_scores) if per_scores else 1)
    if avg_score >= overall_threshold and all(s >= (per_key_threshold * 0.6) for s in per_scores.values()):
        result["keys_matched"] = 1
    else:
        result["keys_matched"] = 0

    return result

#--------------------------------
# Test set written by professional musicians (30 items, ground-truth is rational music theory + aesthetic parameters)
# Each ground-truth is a dict containing the keys to check (mode/tempo/instruments/genre/feel/key)
# For "forbidden items" use "NOT ..." as expected value
#--------------------------------
test_bank = [
    #Basic emotion class
    {"prompt": "我需要一段快乐的背景音乐。", "gt": {"key":"C Major","mode":"major","tempo":"120bpm","instruments":"ukulele, acoustic guitar, light percussion","genre":"acoustic folk","feel":"bright, cheerful, upbeat"}},
    {"prompt": "来点悲伤的音乐。", "gt": {"key":"A minor","mode":"minor","tempo":"65bpm","instruments":"solo piano, slow strings","genre":"cinematic","feel":"somber, melancholic, reflective"}},
    {"prompt": "我感觉很平静，很放松。", "gt": {"mode":"major","tempo":"60bpm","instruments":"ambient synth pad, gentle electric piano","genre":"ambient","feel":"calm, serene, meditative"}},
    {"prompt": "愤怒的，非常有力的。", "gt": {"key":"D minor","mode":"minor","tempo":"140bpm","instruments":"distorted electric guitar, heavy drums, orchestral hits","genre":"rock","feel":"angry, intense, powerful"}},
    {"prompt": "孤独的，空旷的。", "gt": {"mode":"minor","tempo":"55bpm","instruments":"solo cello, distant piano, light wind sound","genre":"ambient","feel":"lonely, empty, desolate"}},

    #complex emotion
    {"prompt": "紧张但又有点暧昧的感觉。", "gt": {"mode":["phrygian","minor"],"tempo":"120bpm","instruments":"pizzicato strings, low synth bass, subtle high-hat","genre":"suspense","feel":"tense, ambiguous, seductive"}},
    {"prompt": "悲伤，但又带着一丝希望。", "gt": {"mode":"minor","tempo":"75bpm","instruments":"piano melody, underlying string pad, flute counter-melody","genre":"cinematic","feel":"melancholic but hopeful, bittersweet"}},
    {"prompt": "平静中带着一点点不安。", "gt": {"mode":"minor","tempo":"60bpm","instruments":"ambient pad, single piano notes, distant dissonant chime","genre":"ambient","feel":"calm but unsettling, uneasy"}},
    {"prompt": "怀旧的，但快乐的童年回忆。", "gt": {"key":"F Major","mode":"major","tempo":"100bpm","instruments":"music box, glockenspiel, light acoustic guitar","genre":"acoustic","feel":"nostalgic, happy, innocent"}},

    # Vlogger scene
    {"prompt": "早晨起床冲咖啡的Vlog。", "gt": {"mode":"major","tempo":"90bpm","instruments":"acoustic guitar, light piano, snaps, soft shakers","genre":"acoustic pop","feel":"calm, positive, morning, gentle"}},
    {"prompt": "东京夜景的旅行 Vlog，需要夜晚 chill 的感觉。", "gt": {"key":"A minor","mode":"minor","tempo":"85bpm","instruments":"lo-fi electric piano, muffled drum beat, synth pad, vinyl crackle","genre":"lo-fi hip hop","feel":"urban, chill, nighttime, stylish"}},
    {"prompt": "健身房锻炼，需要有活力的BGM。", "gt": {"mode":"minor","tempo":"125bpm","instruments":"synth bass, strong electronic drums, aggressive synth lead","genre":"EDM","feel":"energetic, motivational, powerful"}},
    {"prompt": "美食Vlog，展示食物特写。", "gt": {"mode":"major","tempo":"100bpm","instruments":"upright bass, jazz piano, brushed drums, light saxophone","genre":"jazz","feel":"playful, happy, delicious"}},

    #podcast/narrative
    {"prompt": "真实犯罪播客，氛围要紧张且专业。", "gt": {"mode":"minor","tempo":"80bpm","instruments":"low synth drone, distant piano, sparse percussion","genre":"dark ambient","feel":"suspenseful, mysterious, investigative"}},
    {"prompt": "太空探索科普，富有想象力。", "gt": {"mode":"dorian","tempo":"90bpm","instruments":"ambient synth pads, arpeggiated synth, subtle electronic beat","genre":"ambient electronic","feel":"wonder, curious, futuristic, vast"}},
    {"prompt": "情感故事播客，讲述感人的重逢。", "gt": {"key":"E-flat Major","mode":"major","tempo":"70bpm","instruments":"solo piano, high strings crescendo, cello","genre":"cinematic","feel":"touching, emotional, heartwarming, hopeful"}},

    # Dynamic/turning scenes
    {"prompt": "一段音乐，从平缓寂寥开始，然后在28秒后转为紧张。", "gt": {"mode":"minor","tempo":"60bpm then 120bpm","instruments":"ambient pad -> pizzicato strings, low synth pulse","genre":"ambient -> suspense","feel":"calm, lonely, then tense"}},
    {"prompt": "从悲伤转为坚定，情绪有转折。", "gt": {"mode":"minor","tempo":"100bpm","instruments":"piano, full string section, subtle timpani roll","genre":"cinematic","feel":"transitioning, resolving, determined"}},

    #Stylization /sense of the times
    {"prompt": "80年代复古流行，但不要有真实鼓声（不要鼓）。", "gt": {"genre":"synthwave","instruments":["analog synth lead, synth bass, reverb guitar","NOT drums"],"tempo":"120bpm","feel":"retro, 80s, nostalgic, cool"}},
    {"prompt": "中世纪幻想风格，像在酒馆里。", "gt": {"mode":"dorian","tempo":"110bpm","instruments":"lute, acoustic guitar, fiddle, hand drum","genre":"fantasy folk","feel":"medieval, tavern, cheerful, rustic"}},

    # Scene details /timbre constraints
    {"prompt": "一个宁静的睡前冥想段，大约20秒，无鼓。", "gt": {"mode":"lydian","tempo":"50bpm","instruments":"ambient synth pad, crystal bowls, no beat","genre":"meditation","feel":"minimalist, peaceful, ethereal, healing"}},
    {"prompt": "儿童故事播客， whimsical and magical。", "gt": {"key":"C Major","mode":"major","tempo":"110bpm","instruments":"glockenspiel, pizzicato strings, flute, light percussion","genre":"childrens","feel":"magical, happy, whimsical, innocent"}},

    # Scene emotion combination
    {"prompt": "自由奔放的公路驾驶BGM，需要 open-road 的感觉。", "gt": {"key":"E Major","mode":"major","tempo":"120bpm","instruments":"acoustic guitar, driving drum beat, bass guitar, harmonica","genre":"folk rock","feel":"free, driving, adventurous, open road"}},
    {"prompt": "史诗性的开场，宏大而带忧郁。", "gt": {"key":"D minor","mode":"minor","tempo":"140bpm","instruments":"full orchestra, choir, timpani, brass fanfare","genre":"epic orchestral","feel":"grand, heroic, melancholic"}},

    #niche/style mix
    {"prompt": "赛博朋克城市夜景，需要冷峻但有律动。", "gt": {"key":"F# minor","mode":"minor","tempo":"140bpm","instruments":"heavy synth bass, driving electronic drums, neon synth lead","genre":"synthwave","feel":"cyberpunk, futuristic, dark, energetic"}},
    {"prompt": "8-bit 游戏风格，欢快。", "gt": {"key":"C Major","mode":"major","tempo":"130bpm","instruments":"chiptune synth lead, 8-bit drums, square wave bass","genre":"chiptune","feel":"retro, happy, video game, nostalgic"}},

    # Negation/boundary cases
    {"prompt": "我要一段背景音乐，但不希望有任何人声或人声采样（必须无声）。", "gt": {"instruments":"NOT vocals","genre":"ambient","feel":"instrumental, background"}},
    {"prompt": "一个很酷的侦探主题，爵士风。", "gt": {"key":"A minor","mode":"minor","tempo":"95bpm","instruments":"walking bassline, ride cymbal, moody saxophone, jazz piano","genre":"jazz noir","feel":"cool, mysterious, detective, smoky"}},

    # Comprehensive review at the end (combine intention and duration)
    {"prompt": "我需要一段2分03秒的可循环背景音乐，气氛安静但略带忧郁。", "gt": {"tempo":"~60bpm","instruments":"solo piano, ambient pad, distant strings","genre":"cinematic","feel":"quiet, melancholic, loopable"}},
    {"prompt": "片段需要短促、轻快、适合 5 秒的转场 B-roll，感觉俏皮。", "gt": {"tempo":"130bpm","instruments":"xylophone, pizzicato strings, light percussion","genre":"comedy","feel":"playful, quirky, fun"}}
]

# -----------------------------
# Evaluation process: call the model, parse, score, and record the comparison one by one
# -----------------------------
rows = []
start_time = time.time()
for idx, item in enumerate(test_bank, start=1):
    prompt = item["prompt"]
    gt = item["gt"]
    # Prepare an expected_keys list of tuples (key, expected_value) for evaluator
    expected_keys = []
    # Flatten gt dict into expected_keys list
    for k, v in gt.items():
        expected_keys.append((k, v))
    # Model response
    resp = get_model_response(prompt)
    # Evaluate
    scores = evaluate_response_improved(resp, expected_keys, per_key_threshold=0.6, overall_threshold=0.7)
    # Collect display-friendly ground truth (as string) and parsed model json if any
    parsed_model = scores.get("parsed_json", None)
    rows.append({
        "idx": idx,
        "prompt": prompt,
        "ground_truth": json.dumps(gt, ensure_ascii=False),
        "model_response_raw": resp,
        "model_parsed_json": json.dumps(parsed_model, ensure_ascii=False) if parsed_model else None,
        "json_valid": scores["json_valid"],
        "per_key_scores": scores["per_key_scores"],
        "keys_matched": scores["keys_matched"]
    })
    # Print per-case summary for convenience
    print(f"\n[{idx}/{len(test_bank)}] Prompt: {prompt}")
    print("Ground-truth:", json.dumps(gt, ensure_ascii=False))
    print("Model raw response:", resp)
    print("Parsed JSON:", parsed_model)
    print("Per-key scores:", scores["per_key_scores"])
    print("Pass keys_matched:", bool(scores["keys_matched"]))

total_time = time.time() - start_time

# -----------------------------
# Summarize and calculate accuracy
# -----------------------------
df = pd.DataFrame(rows)
json_valid_rate = df["json_valid"].mean()
keys_matched_rate = df["keys_matched"].mean()

# Average score per key (expand all per_key_scores)
# Collect all keys from test_bank to compute per-key average across dataset
all_keys = set()
for t in test_bank:
    for k in t["gt"].keys():
        all_keys.add(k)
all_keys = sorted(list(all_keys))

# compute average score per key (where present)
key_avg_scores = {}
for k in all_keys:
    vals = []
    for r in rows:
        scores_dict = r["per_key_scores"]
        if k in scores_dict:
            vals.append(scores_dict[k])
    key_avg_scores[k] = (sum(vals)/len(vals)) if vals else float("nan")

# Present summary
print("\n" + "="*60)
print("评估完成。用时: {:.1f}s | 测试条数: {}".format(total_time, len(test_bank)))
print(f"JSON 格式合规率: {json_valid_rate:.2%}")
print(f"整体按键匹配率 (keys_matched): {keys_matched_rate:.2%}")
print("每项 key 的平均得分:")
for k,v in key_avg_scores.items():
    print(f"  {k}: {v:.3f}")
print("="*60)

# Display the DataFrame (will be displayed in tabular form in Colab)
pd.set_option('display.max_colwidth', 400)
display_cols = ["idx","prompt","ground_truth","model_response_raw","model_parsed_json","json_valid","per_key_scores","keys_matched"]
display_df = df[display_cols]
display_df.reset_index(drop=True, inplace=True)
display(display_df)

# Optional: Save as CSV for offline viewing
display_df.to_csv("evaluation_results.csv", index=False, encoding="utf-8-sig")
print("已将结果保存为 evaluation_results.csv")



[1/30] Prompt: 我需要一段快乐的背景音乐。
Ground-truth: {"key": "C Major", "mode": "major", "tempo": "120bpm", "instruments": "ukulele, acoustic guitar, light percussion", "genre": "acoustic folk", "feel": "bright, cheerful, upbeat"}
Model raw response: {'key': 'C Major', 'mode': 'major', 'tempo': '120bpm', 'instruments': 'ukulele, acoustic guitar, light percussion', 'genre': 'acoustic folk', 'feel': 'bright, cheerful, upbeat'}
Parsed JSON: {'key': 'C Major', 'mode': 'major', 'tempo': '120bpm', 'instruments': 'ukulele, acoustic guitar, light percussion', 'genre': 'acoustic folk', 'feel': 'bright, cheerful, upbeat'}
Per-key scores: {'key': 1.0, 'mode': 1.0, 'tempo': 1.0, 'instruments': 1.0, 'genre': 1.0, 'feel': 1.0}
Pass keys_matched: True

[2/30] Prompt: 来点悲伤的音乐。
Ground-truth: {"key": "A minor", "mode": "minor", "tempo": "65bpm", "instruments": "solo piano, slow strings", "genre": "cinematic", "feel": "somber, melancholic, reflective"}
Model raw response: {'key': 'A minor', 'mode': 'minor', 'temp

,idx,prompt,ground_truth,model_response_raw,model_parsed_json,json_valid,per_key_scores,keys_matched
0,1,我需要一段快乐的背景音乐。,"{""key"": ""C Major"", ""mode"": ""major"", ""tempo"": ""120bpm"", ""instruments"": ""ukulele, acoustic guitar, light percussion"", ""genre"": ""acoustic folk"", ""feel"": ""bright, cheerful, upbeat""}","{'key': 'C Major', 'mode': 'major', 'tempo': '120bpm', 'instruments': 'ukulele, acoustic guitar, light percussion', 'genre': 'acoustic folk', 'feel': 'bright, cheerful, upbeat'}","{""key"": ""C Major"", ""mode"": ""major"", ""tempo"": ""120bpm"", ""instruments"": ""ukulele, acoustic guitar, light percussion"", ""genre"": ""acoustic folk"", ""feel"": ""bright, cheerful, upbeat""}",1,"{'key': 1.0, 'mode': 1.0, 'tempo': 1.0, 'instruments': 1.0, 'genre': 1.0, 'feel': 1.0}",1
1,2,来点悲伤的音乐。,"{""key"": ""A minor"", ""mode"": ""minor"", ""tempo"": ""65bpm"", ""instruments"": ""solo piano, slow strings"", ""genre"": ""cinematic"", ""feel"": ""somber, melancholic, reflective""}","{'key': 'A minor', 'mode': 'minor', 'tempo': '65bpm', 'instruments': 'solo piano, slow strings', 'genre': 'cinematic', 'feel': 'somber, melancholic, reflective'}","{""key"": ""A minor"", ""mode"": ""minor"", ""tempo"": ""65bpm"", ""instruments"": ""solo piano, slow strings"", ""genre"": ""cinematic"", ""feel"": ""somber, melancholic, reflective""}",1,"{'key': 1.0, 'mode': 1.0, 'tempo': 1.0, 'instruments': 1.0, 'genre': 1.0, 'feel': 1.0}",1
2,3,我感觉很平静，很放松。,"{""mode"": ""major"", ""tempo"": ""60bpm"", ""instruments"": ""ambient synth pad, gentle electric piano"", ""genre"": ""ambient"", ""feel"": ""calm, serene, meditative""}","{'key': 'E Major', 'mode': 'major', 'tempo': '60bpm', 'instruments': 'ambient synth pad, gentle piano, light snaps', 'genre': 'ambient', 'feel': 'calm, peaceful, relaxed'}","{""key"": ""E Major"", ""mode"": ""major"", ""tempo"": ""60bpm"", ""instruments"": ""ambient synth pad, gentle piano, light snaps"", ""genre"": ""ambient"", ""feel"": ""calm, peaceful, relaxed""}",1,"{'mode': 1.0, 'tempo': 1.0, 'instruments': 0.6589285714285714, 'genre': 1.0, 'feel': 0.2804255319148936}",0
3,4,愤怒的，非常有力的。,"{""key"": ""D minor"", ""mode"": ""minor"", ""tempo"": ""140bpm"", ""instruments"": ""distorted electric guitar, heavy drums, orchestral hits"", ""genre"": ""rock"", ""feel"": ""angry, intense, powerful""}","{'key': 'D minor', 'mode': 'minor', 'tempo': '140bpm', 'instruments': 'distorted electric guitar, heavy drums, orchestral hits', 'genre': 'rock', 'feel': 'angry, intense, powerful'}","{""key"": ""D minor"", ""mode"": ""minor"", ""tempo"": ""140bpm"", ""instruments"": ""distorted electric guitar, heavy drums, orchestral hits"", ""genre"": ""rock"", ""feel"": ""angry, intense, powerful""}",1,"{'key': 1.0, 'mode': 1.0, 'tempo': 1.0, 'instruments': 1.0, 'genre': 1.0, 'feel': 1.0}",1
4,5,孤独的，空旷的。,"{""mode"": ""minor"", ""tempo"": ""55bpm"", ""instruments"": ""solo cello, distant piano, light wind sound"", ""genre"": ""ambient"", ""feel"": ""lonely, empty, desolate""}","{'key': 'F minor', 'mode': 'minor', 'tempo': '55bpm', 'instruments': 'solo cello, distant piano, light wind sound', 'genre': 'ambient', 'feel': 'lonely, empty, desolate'}","{""key"": ""F minor"", ""mode"": ""minor"", ""tempo"": ""55bpm"", ""instruments"": ""solo cello, distant piano, light wind sound"", ""genre"": ""ambient"", ""feel"": ""lonely, empty, desolate""}",1,"{'mode': 1.0, 'tempo': 1.0, 'instruments': 1.0, 'genre': 1.0, 'feel': 1.0}",1
5,6,紧张但又有点暧昧的感觉。,"{""mode"": [""phrygian"", ""minor""], ""tempo"": ""120bpm"", ""instruments"": ""pizzicato strings, low synth bass, subtle high-hat"", ""genre"": ""suspense"", ""feel"": ""tense, ambiguous, seductive""}","{'key': 'D minor', 'mode': 'minor', 'tempo': '120bpm', 'instruments': 'pizzicato strings, low synth bass, subtle high-hat', 'genre': 'suspense', 'feel': 'tense, ambiguous, seductive'}","{""key"": ""D minor"", ""mode"": ""minor"", ""tempo"": ""120bpm"", ""instruments"": ""pizzicato strings, low synth bass, subtle

已将结果保存为 evaluation_results.csv


In [ ]:
# ----------------------------------------------------------------------
# Cell 6: (Optional) Mount Google Drive to save the model
# ----------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive 已挂载到 /content/drive 目录")

Mounted at /content/drive
Google Drive 已挂载到 /content/drive 目录


In [ ]:
"""
Cell 7: Model Artifact Persistence

This cell serializes the fine-tuned LoRA adapter weights and the associated
tokenizer to permanent storage (Google Drive). This allows the specific
fine-tuned behaviors to be reloaded later without retraining the base model.
"""

import os

# 1. Define Output Directory
# We save to Google Drive to ensure artifacts persist after the Colab runtime disconnects.
save_path = "/content/drive/MyDrive/AuraFlow/best_llm_adapter_r32_e3"

print(f"--- Saving artifacts to: {save_path} ---")

# 2. Save LoRA Adapter
# This saves the specific Low-Rank Adaptation weights (adapter_model.safetensors)
# and the configuration file. It does NOT save the full base model, keeping the file size small.
trainer.model.save_pretrained(save_path)

# 3. Save Tokenizer
# Saving the tokenizer ensures that the vocabulary and special tokens remain
# consistent during the inference phase.
trainer.tokenizer.save_pretrained(save_path)

print("--- Save Complete ---")
print(f"The LoRA adapter has been successfully archived to Google Drive.")

--- 成功! ---
你训练的 LoRA 适配器已永久保存到你的 Google Drive:
/content/drive/MyDrive/AuraFlow/best_llm_adapter_r32_e3
